### Path to access this notebook
* %run /Users/Weiqi0/ISB_working/Hadlock_lab/QI_ISB_Git_repo/TranslatorPharcogenomicsKG/Parser_helper_functions.ipynb

In [1]:
## Load needed packages
import uuid
import pandas as pd
import re

In [2]:
def generate_uuid(row):
    """
    Generates a UUID based on the combined values of multiple columns.
    """
    combined_string = ''.join(row.astype(str))
    return uuid.uuid5(uuid.NAMESPACE_DNS, combined_string)

In [3]:
def generate_uuid_from_columns(df, column_list, namespace=uuid.NAMESPACE_DNS):
    """
    Generates UUIDs based on the values in a specified column of a Pandas DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_list (list): List of all names of columns to use for UUID generation.
        namespace (uuid.UUID): A UUID namespace (default is uuid.NAMESPACE_DNS).

    Returns:
        pd.Series: A Pandas Series containing the generated UUIDs.
    """
    return df[column_list].apply(lambda x: uuid.uuid5(namespace, str(x)).hex)

In [4]:
def add_ensembl_curie_columns(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """
    Takes a DataFrame and the name of a column containing Ensembl transcript IDs
    (with or without version suffix, e.g. "ENST00000381652.3" or "ENST00000381652"),
    and returns the DataFrame with two new columns:
      - ensembl_identifier: the CURIE with version stripped, e.g. "ENSEMBL:ENST00000381652"
      - ensembl_id_version: the version string, e.g. "3" (or None if absent/missing)
    """

    def to_ensembl_curie(ensembl_id_with_version) -> tuple[str, str]:
        """Returns (curie, version) — version stripped for the canonical CURIE."""
        if pd.isna(ensembl_id_with_version):
            return None, None
        ensembl_id_with_version = str(ensembl_id_with_version).strip()
        if ensembl_id_with_version == "":
            return None, None
        match = re.match(r"^(ENST\d+)\.(\d+)$", ensembl_id_with_version)
        if match:
            stable_id, version = match.groups()
            return f"ENSEMBL:{stable_id}", version
        return f"ENSEMBL:{ensembl_id_with_version}", None

    df = df.copy()
    df[["ensembl_identifier", "ensembl_id_version"]] = df[column_name].apply(
        lambda val: pd.Series(to_ensembl_curie(val))
    )
    return df